> 📂 **Practice Set 4: Graphs, Dynamic Programming and Greedy** · [assignments](../README.md) · [course home](../../README.md)

# Practice Set 4: Graphs, Dynamic Programming and Greedy (Solutions)

**Covers:** Handbook chapters 22, 25 and 26

Traversals and orderings on graphs, tables for dynamic programming, and a greedy rule.

Compare with your own attempt. Try each problem yourself first!

## Setup

Run this cell once. It prepares what the problems and their checks need (for example the `check_recursive` helper or a `Node` class).

In [1]:
# Nothing to prepare for this set. Graphs are dictionaries that map a node to a list of neighbours.


## Problem 1: Count the islands

Write `count_islands(grid)`: `grid` is a list of strings of `'1'` (land) and `'0'` (water). Count the groups of land connected up, down, left or right.

In [2]:
def count_islands(grid):
    rows, cols = len(grid), len(grid[0])
    seen = set()

    def sink(r, c):
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != '1' or (r, c) in seen:
            return
        seen.add((r, c))
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            sink(r + dr, c + dc)

    islands = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1' and (r, c) not in seen:
                islands += 1
                sink(r, c)
    return islands

In [3]:
assert count_islands(['11000', '11000', '00100', '00011']) == 3
assert count_islands(['000']) == 0 and count_islands(['111', '101', '111']) == 1
print('✅ Problem 1 passed')

✅ Problem 1 passed


## Problem 2: Shortest path length

Write `shortest_path_length(graph, start, goal)`: the fewest edges from `start` to `goal`, or -1 if unreachable. Use BFS.

In [4]:
from collections import deque


def shortest_path_length(graph, start, goal):
    seen = {start}
    queue = deque([(start, 0)])
    while queue:
        node, steps = queue.popleft()
        if node == goal:
            return steps
        for neighbour in graph[node]:
            if neighbour not in seen:
                seen.add(neighbour)
                queue.append((neighbour, steps + 1))
    return -1

In [5]:
g = {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A', 'D'], 'D': ['B', 'C', 'E'], 'E': ['D'], 'F': []}
assert shortest_path_length(g, 'A', 'E') == 3
assert shortest_path_length(g, 'A', 'A') == 0
assert shortest_path_length(g, 'A', 'F') == -1
print('✅ Problem 2 passed')

✅ Problem 2 passed


## Problem 3: Topological order

Write `topological_order(graph)`: `graph` maps each node to the nodes that must come **after** it. Return a valid order, or `None` if there is a cycle.

In [6]:
from collections import deque


def topological_order(graph):
    in_degree = {node: 0 for node in graph}
    for node in graph:
        for later in graph[node]:
            in_degree[later] += 1
    ready = deque(node for node, d in in_degree.items() if d == 0)
    order = []
    while ready:
        node = ready.popleft()
        order.append(node)
        for later in graph[node]:
            in_degree[later] -= 1
            if in_degree[later] == 0:
                ready.append(later)
    return order if len(order) == len(graph) else None

In [7]:
dag = {'Python': ['DSA', 'Web'], 'DSA': ['ML'], 'Maths': ['ML'], 'Web': [], 'ML': []}
order = topological_order(dag)
assert sorted(order) == sorted(dag)
position = {node: i for i, node in enumerate(order)}
assert all(position[a] < position[b] for a in dag for b in dag[a])
assert topological_order({'A': ['B'], 'B': ['A']}) is None and topological_order({}) == []
print('✅ Problem 3 passed')

✅ Problem 3 passed


## Problem 4: Fewest coins

Write `min_coins(coins, amount)`: the fewest coins that sum to `amount` (each coin may be reused), or -1. Careful: greedy fails for `[1, 3, 4]`.

In [8]:
def min_coins(coins, amount):
    INF = float('inf')
    fewest = [0] + [INF] * amount
    for a in range(1, amount + 1):
        for coin in coins:
            if coin <= a and fewest[a - coin] + 1 < fewest[a]:
                fewest[a] = fewest[a - coin] + 1
    return fewest[amount] if fewest[amount] != INF else -1

In [9]:
assert min_coins([1, 3, 4], 6) == 2 and min_coins([1, 2, 5], 11) == 3
assert min_coins([2], 3) == -1 and min_coins([5], 0) == 0
print('✅ Problem 4 passed')

✅ Problem 4 passed


## Problem 5: Longest common subsequence

Write `lcs_length(a, b)`: the length of the longest common subsequence of two strings.

In [10]:
def lcs_length(a, b):
    table = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                table[i][j] = 1 + table[i - 1][j - 1]
            else:
                table[i][j] = max(table[i - 1][j], table[i][j - 1])
    return table[-1][-1]

In [11]:
assert lcs_length('ABCBDAB', 'BDCABA') == 4
assert lcs_length('', 'abc') == 0 and lcs_length('abc', 'abc') == 3 and lcs_length('abc', 'xyz') == 0
print('✅ Problem 5 passed')

✅ Problem 5 passed


## Problem 6: Schedule the most meetings

Write `max_meetings(intervals)`: the most non-overlapping `(start, end)` meetings (a meeting may start exactly when another ends). Which sort key makes greedy correct?

In [12]:
def max_meetings(intervals):
    count, last_end = 0, float('-inf')
    for start, end in sorted(intervals, key=lambda iv: iv[1]):
        if start >= last_end:
            count += 1
            last_end = end
    return count

In [13]:
assert max_meetings([(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]) == 4
assert max_meetings([(0, 10), (1, 2), (3, 4)]) == 2 and max_meetings([]) == 0
print('✅ Problem 6 passed')

✅ Problem 6 passed


## Problem 7: House robber

Write `rob(houses)`: the largest total you can take from a row of houses without taking two neighbours.

In [14]:
def rob(houses):
    skip, take = 0, 0
    for money in houses:
        skip, take = take, max(take, skip + money)
    return take

In [15]:
assert rob([2, 7, 9, 3, 1]) == 12 and rob([1, 2, 3, 1]) == 4
assert rob([]) == 0 and rob([5]) == 5
print('✅ Problem 7 passed')

✅ Problem 7 passed
